# Climate Policy Agent — CSI 6007, Assignment C

This notebook runs the same `agent.py` logic as the repository website. The agent classifies a question, retrieves facts from the assigned article, chooses a skill, validates citations and refuses unsupported questions. **No GPU is needed.**

The article discusses PM2.5 changes between La Niña winter 2022–23 and El Niño winter 2023–24, plus long-term climate-zone shifts and farming implications. The findings are limited: only two unusual winters were compared, and correlation does not prove causation.

**Run from top to bottom.** Default mode works without an API key. Optional Groq synthesis is only enabled if you configure a Colab secret named `GROQ_API_KEY`. Never put a secret in a notebook cell or GitHub repository.

In [ ]:
# Option A: If you opened this notebook in Colab, clone your own GitHub repository.
# Replace YOUR-USERNAME with your GitHub username after uploading the repo.
import os, sys, subprocess
REPO_URL = "https://github.com/YOUR-USERNAME/climate-policy-agent.git"
if not os.path.exists("climate-policy-agent"):
    result = subprocess.run(["git", "clone", REPO_URL, "climate-policy-agent"], capture_output=True, text=True)
    if result.returncode != 0:
        print("Clone did not run (perhaps the repo is not uploaded yet). Upload the repository ZIP to Colab or edit REPO_URL.")
        print(result.stderr[-600:])
REPO = os.path.abspath("climate-policy-agent")
if os.path.isdir(REPO):
    os.chdir(REPO)
    sys.path.insert(0, REPO)
    print("Repository:", REPO)
else:
    print("Current directory:", os.getcwd())

In [ ]:
# Install the agent's optional dependencies. The evidence-only agent itself uses the Python standard library.
%pip -q install groq mcp pytest

In [ ]:
# Import and inspect the evidence store.
import json
from pathlib import Path
import agent
facts = agent.load_facts()
print(f"Loaded {len(facts)} article facts")
for f in facts[:3]:
    print(f"{f['id']} · {f['topic']}: {f['fact']}")

## 1. Run the agent without an LLM

This is the safest reproducible mode: answers are based on the article facts and unsupported questions should be refused.

In [ ]:
questions = [
    "Why did Delhi and Jaipur become more polluted in winter 2023-24?",
    "What policy options should governments consider?",
    "How could climate-zone shifts affect farmers in Uttar Pradesh?",
    "Who won the football World Cup?",  # Should refuse: outside the assigned article
]
for q in questions:
    print("\n" + "="*88 + "\nQUESTION:", q)
    print(json.dumps(agent.run_agent(q, use_llm=False), indent=2, ensure_ascii=False))

## 2. Optional LLM synthesis (API key is optional)

Only use this in your own trusted Colab session. Add a Colab secret called `GROQ_API_KEY` using the key icon in the left sidebar and grant this notebook access. The key is read from the secret store and is never written to a file. If you do not have a key, skip this cell; evidence-only mode above still works.

In [ ]:
# Optional: set up a Colab secret without printing the secret.
USE_GROQ = False
try:
    from google.colab import userdata
    key = userdata.get("GROQ_API_KEY")
    if key:
        os.environ["GROQ_API_KEY"] = key
        USE_GROQ = True
except Exception:
    pass
print("Optional LLM configured:", USE_GROQ)
if USE_GROQ:
    q = "Why might local emission cuts not guarantee cleaner air in Delhi?"
    print(json.dumps(agent.run_agent(q, use_llm=True), indent=2, ensure_ascii=False))
else:
    print("Skipped. Evidence-only agent is ready; no API key is required.")

## 3. Agent tools / MCP tool logic

The same tool functions are available to the MCP server in `mcp_server.py`. The server itself is normally launched by an MCP-capable client on a local machine; Colab can test the tool functions directly.

In [ ]:
print("SEARCH TOOL")
print(json.dumps(agent.tool_search_article_facts("wind and pollution movement", limit=4), indent=2, ensure_ascii=False))
print("\nWINTER COMPARISON TOOL")
comparison = agent.tool_compare_winter_patterns()
print("La Niña fact IDs:", [f["id"] for f in comparison["la_nina_2022_23"]])
print("El Niño fact IDs:", [f["id"] for f in comparison["el_nino_2023_24"]])
print("Caveat:", comparison["caveat"]["fact"])
print("\nEVIDENCE VALIDATION TOOL")
print(agent.tool_check_evidence(["F1", "F10", "F999"]))

## 4. Run automated tests

These tests check retrieval, refusal, policy trade-offs and evidence-ID validation. A successful test run is useful demo evidence; do not claim tests passed unless you see the result below.

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Tests failed; inspect the output above.")
print("All tests passed.")